# Sesión 2. Ancestría global: PCA y ADMIXTURE

En la sesión 1 dejamos los datos listos: 969 personas y 59 850 variantes con control de calidad y poda por LD. Ahora vamos a estimar la **ancestría global**: qué proporción del genoma de cada persona viene de cada población ancestral.

Pasos:

1. Rehacer los datos de la sesión 1
2. Análisis de componentes principales (PCA)
3. Correr ADMIXTURE
4. Graficar e interpretar los resultados

Después de cada paso hay una tabla con los archivos que crea. El resumen completo está en [archivos_por_paso.md](https://github.com/YTZ27/curso-ancestria/blob/main/notebooks/archivos_por_paso.md).

## 1. Rehacer los datos de la sesión 1

Colab borra los archivos al cerrar el cuaderno, así que volvemos a generarlos. Es exactamente lo mismo que hicimos la sesión pasada, en una sola celda: descargar PLINK, descargar el VCF, convertirlo a binario, filtrar por calidad y podar por LD.

Tarda menos de un minuto. Si quieres repasar qué hace cada comando, está explicado paso a paso en el cuaderno de la sesión 1.

In [ ]:
%%bash
wget -nv -O plink2.zip https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_x86_64_20261015.zip && unzip -o -q plink2.zip
wget -nv -nc https://github.com/YTZ27/curso-ancestria/raw/main/data/exome.vcf.gz

./plink2 --vcf exome.vcf.gz --make-bed --out exome_data
./plink2 --bfile exome_data --maf 0.01 --geno 0.1 --mind 0.1 --make-bed --out qc_maf-0.01_geno-0.1
./plink2 --bfile qc_maf-0.01_geno-0.1 --indep-pairwise 50 5 0.2 --out ld_pruned
./plink2 --bfile qc_maf-0.01_geno-0.1 --extract ld_pruned.prune.in --make-bed --out admixture_LD_QC

In [ ]:
!ls -lh admixture_LD_QC.*
!echo "Personas:  $(wc -l < admixture_LD_QC.fam)"
!echo "Variantes: $(wc -l < admixture_LD_QC.bim)"

**Archivos de este paso**

| Archivo | Qué es | ¿Se usa después? |
|---|---|---|
| `admixture_LD_QC.bed / .bim / .fam` | Datos con control de calidad y poda por LD: 969 personas, 59 850 variantes | Sí: entrada del PCA (paso 2) y de ADMIXTURE (paso 3). El `.fam` además identifica a cada persona en los gráficos |
| `exome.vcf.gz, exome_data.*, qc_*, ld_pruned.*` | Los archivos intermedios, explicados en la sesión 1 | No a partir de aquí |

## 2. Análisis de componentes principales (PCA)

El PCA resume las diferencias genéticas entre todas las personas en unos pocos ejes nuevos, los componentes principales (PCs). Las personas con ancestría parecida quedan cerca unas de otras.

Tiene dos ventajas sobre ADMIXTURE: es casi instantáneo y **no necesita que elijamos un número de poblaciones ancestrales**. Por eso conviene mirarlo primero: muestra la estructura de los datos sin que nosotros le impongamos un modelo.

In [ ]:
!./plink2 --bfile admixture_LD_QC --pca 10 --out pca
!ls -lh pca.*

**Archivos de este paso**

| Archivo | Qué es | ¿Se usa después? |
|---|---|---|
| `pca.eigenvec` | Una fila por persona (`#FID`, `IID`) con su posición en PC1 a PC10 | Sí: para el gráfico de PC1 vs PC2 |
| `pca.eigenval` | 10 números, uno por PC: cuánta variación captura cada uno | Sí: para el gráfico de eigenvalores |
| `pca.log` | Registro del comando | No |

In [ ]:
!head -3 pca.eigenvec
!cat pca.eigenval

¿Cuántos PCs importan? Los primeros capturan casi toda la estructura poblacional; después, los valores se aplanan.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

eigenval = pd.read_csv('pca.eigenval', header=None)[0]
ax = eigenval.plot.bar(color='#6b6b6b', figsize=(6, 3), width=0.7)
ax.set_xticklabels(range(1, len(eigenval) + 1), rotation=0)
ax.set(xlabel='Componente principal', ylabel='Eigenvalor', title='Variación que captura cada PC')
for lado in ['top', 'right']:
    ax.spines[lado].set_visible(False)
plt.tight_layout()
plt.show()

Los dos primeros PCs se despegan del resto. Tiene sentido: con 3 grupos continentales en los datos, bastan 2 ejes para separarlos, igual que con 3 puntos en un plano (Patterson et al., 2006).

Ahora leemos los PCs y le agregamos a cada persona su población y, en el caso de las poblaciones de referencia, su continente.

In [ ]:
pcs = pd.read_csv('pca.eigenvec', sep=r'\s+')
pcs['poblacion'] = pcs['IID'].str.split('_').str[0]

continente = {'forReferenceYRI': 'Africana', 'forReferenceESN': 'Africana', 'forReferenceGWD': 'Africana',
              'forReferenceGBR': 'Europea', 'forReferenceIBS': 'Europea', 'forReferenceTSI': 'Europea',
              'forReferenceKaritiana': 'Nativa americana', 'forReferenceSurui': 'Nativa americana',
              'forReferencePima': 'Nativa americana', 'forReferenceMaya': 'Nativa americana',
              'forReferencePEL': 'Nativa americana'}
pcs['continente'] = pcs['poblacion'].map(continente)  # vacío (NaN) en las poblaciones mezcladas
pcs.head()

PC1 contra PC2: las poblaciones de referencia van coloreadas por continente, y cada panel muestra una población mezclada en negro.

In [ ]:
colores = {'Africana': '#eb6834', 'Europea': '#2a78d6', 'Nativa americana': '#1baf7a'}
fig, axes = plt.subplots(2, 2, figsize=(10, 9), sharex=True, sharey=True)

for ax, pob in zip(axes.flat, ['ACB', 'ASW', 'PUR', 'MXL']):
    for nombre, color in colores.items():
        ref = pcs[pcs['continente'] == nombre]
        ax.scatter(ref['PC1'], ref['PC2'], s=14, color=color, alpha=0.5, label=f'Referencia {nombre.lower()}')
    mez = pcs[pcs['poblacion'] == pob]
    ax.scatter(mez['PC1'], mez['PC2'], s=14, color='black', label='Población mezclada')
    ax.set_title(pob)

axes[0, 0].legend(frameon=False, fontsize=9)
for ax in axes[1]:
    ax.set_xlabel('PC1')
for ax in axes[:, 0]:
    ax.set_ylabel('PC2')
plt.tight_layout()
plt.show()

**Cómo leerlo:** cada continente forma su propio grupo, y las personas mezcladas caen *entre* los grupos de sus poblaciones de origen; mientras más cerca está una persona de un grupo, mayor es esa ancestría (McVean, 2009).

El signo de un PC es arbitrario: tu gráfico puede salir volteado respecto al de otra persona, y significa lo mismo.

Los gráficos se muestran aquí y no se guardan como archivos.

## 3. Correr ADMIXTURE

El PCA nos mostró la estructura, pero no le pone número a la ancestría de cada persona. Eso lo hace ADMIXTURE: supone que existen **K** poblaciones ancestrales y estima qué proporción del genoma de cada persona viene de cada una.

Primero lo descargamos.

In [ ]:
!wget -nv -nc https://dalexander.github.io/admixture/binaries/admixture_linux-1.3.0.tar.gz
!tar -xzf admixture_linux-1.3.0.tar.gz
!ls -lh dist/admixture_linux-1.3.0/

**Archivos de este paso**

| Archivo | Qué es | ¿Se usa después? |
|---|---|---|
| `admixture_linux-1.3.0.tar.gz` | La descarga comprimida | No: solo sirve para extraer la carpeta `dist/` |
| `dist/admixture_linux-1.3.0/admixture` | El programa ADMIXTURE | Sí: en la celda siguiente |
| `dist/admixture_linux-1.3.0/admixture-manual.pdf` | El manual de ADMIXTURE | No, pero vale la pena leerlo |

Ahora lo corremos con **K = 3**, porque en estos datos hay tres grupos continentales de referencia: africano, europeo y nativo americano. `-j2` usa los dos procesadores que da Colab. Tarda varios minutos.

In [ ]:
!./dist/admixture_linux-1.3.0/admixture -j2 admixture_LD_QC.bed 3

In [ ]:
!ls -lh admixture_LD_QC.3.*

**Archivos de este paso**

| Archivo | Qué es | ¿Se usa después? |
|---|---|---|
| `admixture_LD_QC.3.Q` | Proporciones de ancestría: una fila por persona (mismo orden que el `.fam`) y una columna por componente ancestral; cada fila suma 1 | Sí: para el gráfico del paso 4, junto con `admixture_LD_QC.fam` para saber de quién es cada fila |
| `admixture_LD_QC.3.P` | Frecuencia alélica de cada variante en cada componente ancestral: una fila por variante (mismo orden que el `.bim`) | No en este cuaderno. Sirve para proyectar personas nuevas sobre los mismos componentes |

El `3` de los nombres es K. ADMIXTURE no escribe archivo `.log`: sus mensajes solo aparecen en la salida de la celda.

In [ ]:
!head -3 admixture_LD_QC.3.Q
!head -3 admixture_LD_QC.fam

## 4. Graficar e interpretar los resultados

Primero leemos los resultados y la población de cada persona (la primera parte de su identificador).

In [ ]:
Q = pd.read_csv('admixture_LD_QC.3.Q', sep=r'\s+', header=None)
fam = pd.read_csv('admixture_LD_QC.fam', sep=r'\s+', header=None)
Q['poblacion'] = fam[1].str.split('_').str[0]
Q.head()

ADMIXTURE no sabe cuál componente es africano, europeo o nativo americano: solo los numera (0, 1, 2), y el orden cambia de una corrida a otra. Para ponerles nombre, miramos en qué población de referencia es más alto cada uno.

In [ ]:
refs = {'forReferenceYRI': 'Africana', 'forReferenceGBR': 'Europea', 'forReferenceKaritiana': 'Nativa americana'}
promedios = Q.groupby('poblacion')[[0, 1, 2]].mean().loc[list(refs)]
promedios.round(2)

In [ ]:
nombres = {k: refs[promedios[k].idxmax()] for k in [0, 1, 2]}
Q = Q.rename(columns=nombres)
nombres

Ahora ordenamos a las personas: primero las poblaciones de referencia (África, Europa, América) y después las mezcladas.

In [ ]:
orden = ['forReferenceYRI', 'forReferenceESN', 'forReferenceGWD',
         'forReferenceGBR', 'forReferenceIBS', 'forReferenceTSI',
         'forReferenceKaritiana', 'forReferenceSurui', 'forReferencePima', 'forReferenceMaya', 'forReferencePEL',
         'ACB', 'ASW', 'PUR', 'MXL']
Q['poblacion'] = pd.Categorical(Q['poblacion'], categories=orden, ordered=True)
Q = Q.sort_values(['poblacion', 'Europea'], ascending=[True, False]).reset_index(drop=True)

Y el gráfico: una barra por persona, agrupadas por población.

In [ ]:
ax = Q[list(colores)].plot.bar(stacked=True, width=1, figsize=(20, 5), color=colores, edgecolor='none')

# Una etiqueta por población, en la mitad de su bloque, y una línea blanca entre bloques
tam = Q.groupby('poblacion', observed=True).size()
fin = tam.cumsum()
ax.set_xticks((fin - tam / 2 - 0.5).values)
ax.set_xticklabels(tam.index.astype(str).str.replace('forReference', 'ref: '), rotation=90)
for x in fin.iloc[:-1]:
    ax.axvline(x - 0.5, color='white', linewidth=1)

ax.set(ylim=(0, 1), xlabel='', ylabel='Proporción de ancestría', title='ADMIXTURE, K = 3')
ax.legend(title='Ancestría', bbox_to_anchor=(1, 1), frameon=False)
for lado in ax.spines.values():
    lado.set_visible(False)
plt.tight_layout()
plt.show()

**Cómo leerlo:** cada barra es una persona y cada color, la proporción de su genoma que viene de ese componente ancestral. Las poblaciones de referencia salen casi de un solo color, y las mezcladas combinan varios.

Comparemos el promedio de cada población mezclada:

In [ ]:
mezcladas = Q[Q['poblacion'].isin(['ACB', 'ASW', 'PUR', 'MXL'])]
mezcladas.groupby('poblacion', observed=True)[list(colores)].mean().round(2)

### Tres advertencias al interpretar

1. **Los componentes no son poblaciones ancestrales reales.** Son grupos estadísticos que dependen de K y de qué poblaciones incluimos. Con otro K o con otras poblaciones, el gráfico cuenta otra historia (Lawson et al., 2018).
2. **K = 3 es una decisión nuestra,** no un resultado. ADMIXTURE puede estimar qué K ajusta mejor con validación cruzada (`--cv`), pero el mejor K estadístico no siempre es el más interpretable.
3. **Ancestría global no es ancestría local.** Decir que alguien tiene 30 % de ancestría nativa americana no dice *qué partes* de su genoma lo son. Eso es lo que veremos en la sesión 3.

El gráfico se muestra aquí y no se guarda. Para guardarlo, clic derecho sobre la imagen, o agrega `plt.savefig('admixture_K3.png', dpi=300)` antes de `plt.show()`.

## Referencias

- Alexander DH, Novembre J, Lange K. Fast model-based estimation of ancestry in unrelated individuals. *Genome Res*. 2009;19:1655–1664. doi:10.1101/gr.094052.109
- Lawson DJ, van Dorp L, Falush D. A tutorial on how not to over-interpret STRUCTURE and ADMIXTURE bar plots. *Nat Commun*. 2018;9:3258. doi:10.1038/s41467-018-05257-7
- McVean G. A genealogical interpretation of principal components analysis. *PLoS Genet*. 2009;5(10):e1000686. doi:10.1371/journal.pgen.1000686
- Patterson N, Price AL, Reich D. Population structure and eigenanalysis. *PLoS Genet*. 2006;2(12):e190. doi:10.1371/journal.pgen.0020190